# Plot runs from `runs/`

Reads every `runs/*/config.json` + `metrics.csv`. Filter with config keys, e.g. `load_runs(dataset="mouse", training_mode="from_scratch")`; a filter value can also be a function.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mouse_model.run_logging import load_runs, plot_runs, summary_table

runs = [r for r in load_runs() if r["summary"]]  # finished runs only
print(len(runs), "finished runs")
for row in summary_table(runs, keys=("dataset", "training_mode", "train_minutes_requested", "seed")):
    print(row)

## Validation correlation vs epoch, one panel per dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, dataset in zip(axes, ["mouse", "sensorium"]):
    sel = [r for r in runs if r["config"].get("dataset") == dataset
           and r["config"].get("training_mode") == "from_scratch"]
    plot_runs(sel, "val_cor", x="epoch", ax=ax, label_keys=("train_minutes_requested", "seed"))
    ax.set_title(f"{dataset}: scratch")
plt.tight_layout()

## Learning curve: correlation at the best-val-loss epoch vs minutes of training data

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for dataset, color in [("mouse", "C0"), ("sensorium", "C1")]:
    sel = [r for r in runs if r["config"].get("dataset") == dataset
           and r["config"].get("training_mode") == "from_scratch"]
    x = np.array([r["config"]["split_info"]["train_minutes"] for r in sel])
    y = np.array([r["summary"]["val_cor_at_best"] for r in sel])
    order = np.argsort(x)
    ax.plot(x[order], y[order], "o-", color=color, label=dataset)
ax.set_xscale("log")
ax.set_xlabel("training data (minutes)")
ax.set_ylabel("val correlation (best val-loss epoch)")
ax.grid(alpha=0.3)
ax.legend()

## Same curves vs gradient steps (separates data size from number of updates)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, dataset in zip(axes, ["mouse", "sensorium"]):
    sel = [r for r in runs if r["config"].get("dataset") == dataset]
    plot_runs(sel, "val_cor", x="step", ax=ax, label_keys=("training_mode", "train_minutes_requested"))
    ax.set_xscale("log")
    ax.set_title(dataset)
plt.tight_layout()

## Per-neuron correlation at the best epoch

In [ ]:
import os

fig, ax = plt.subplots(figsize=(7, 4))
for r in runs:
    p = os.path.join(r["dir"], "per_neuron_best.npz")
    if r["config"].get("train_minutes_requested") is None and os.path.isfile(p):
        cor = np.load(p)["cor_per_neuron"]
        ax.hist(cor[np.isfinite(cor)], bins=50, alpha=0.5, density=True,
                label=f"{r['config']['dataset']} {r['config']['training_mode']}")
ax.set_xlabel("per-neuron val correlation")
ax.legend(fontsize=8)